### `.with_retry()` and `.with_fallbacks()`

**What are they?**

- **`.with_retry()`** – automatically retries a Runnable if it fails (e.g., API timeout, rate limit).
- **`.with_fallbacks()`** – specifies backup Runnables to use if the primary one fails.

**Why use them?**

- Make your chains more **reliable** and **production‑ready**.
- Avoid crashing when a model or API experiences temporary issues.

### How they work

- **Retry:**  
  `chain.with_retry(stop_after_attempt=3, wait_exponential_jitter=True)`
- **Fallback:**  
  `chain.with_fallbacks([fallback_chain])`

If the primary fails, the fallback is tried automatically.

**In This Notebook**

- We create a chain and add retry settings.
- We add a fallback model (gpt-3.5-turbo) in case the primary fails.
- We run the chain normally and observe the output.

Concept: .with_retry() and .with_fallbacks()

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

**stop_after_attempt=3** means: try once, then retry up to 2 more times (total 3 attempts).

**wait_exponential_jitter=True** adds a random wait between retries to avoid thundering herd.

In [3]:
# Load environment variables
load_dotenv()

# Create a model
primary_model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Create a simple prompt
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human', 'Explain {topic} in one sentence.')
])



Using .with_retry()


In [12]:
# Build the base chain
chain = prompt | primary_model | StrOutputParser()

# Create a reliable chain with automatic retries
reliable_chain = chain.with_retry(
    stop_after_attempt=3,
    wait_exponential_jitter=True
)

# Invoke the reliable chain
res = reliable_chain.invoke({'topic': 'RAG'})
print(f'Result:\n{res}')

Result:
RAG, or Retrieval-Augmented Generation, is a natural language processing approach that combines retrieval of relevant documents from a knowledge base with generative models to produce more accurate and contextually rich responses.


 Using .with_fallbacks()

In [13]:
# Create a fallback model
fallback_model = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)

# Build a chain with fallback.
chain_with_fallback = chain.with_fallbacks([
    prompt | fallback_model | StrOutputParser()
])

# Invoke the chain with fallback
result_fallback = chain_with_fallback.invoke({'topic': 'RAG'})

print('Result (with fallback):')
print(result_fallback)

Result (with fallback):
RAG, or Retrieval-Augmented Generation, is a machine learning approach that combines retrieval of relevant information from a knowledge base with generative models to produce more accurate and contextually relevant responses.
